In [76]:
import vertexai
import pandas as pd
from tqdm import tqdm
import json
from google import genai
import os

from search_eval_utils import (
    access_secret_version, 
    download_blob, 
    create_stratified_sample,
    process_and_save_prompt_output,
    prompt_v1, 
    jsonl_to_df
)

### **First generate the `project_df_labeled.csv` and `searches.csv` in the `compare_search_eval.ipynb`**

In [77]:
PROJECT_ID = 'proj-sales-recommender-dev'
LOCATION = 'us-central1'
BUCKET = 'sales_recommender_dev_bucket'
DATASET_NAME = "data/project_df_labeled.csv" # Generated in compare_search_eval.ipynb
SEARCHES_DATASET_NAME = "data/searches.csv" # Generated in compare_search_eval.ipynb
SECRET_NAME = "gemini_api_key"
MODEL_ID = "gemini-2.0-flash-001"

In [78]:
vertexai.init(project=PROJECT_ID, location=LOCATION)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "key.json"
os.environ["GEMINI_API_KEY"] = access_secret_version(PROJECT_ID, SECRET_NAME)
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [79]:
# Fetch and sample the dataframe for evaluation 

project_df_labeled = pd.read_csv(DATASET_NAME)

sampled_df = create_stratified_sample(
    project_df_labeled, 
    n_yes = 10, 
    n_no = 10
)

sampled_df_no_label = sampled_df.drop(columns=['True_Search_Label'])

In [80]:
#  Filters_dict holds DataFrames with each search term and its corresponding (new) query

filters_dict = {}

filters_dict['Ceilings'] = pd.DataFrame({"SEARCH": ["Ceilings"], "Query": ['''
( Title:(ceiling* AND (replacement OR repair OR installation OR upgrade OR grid))) OR \
( (armstrong NEAR ceiling*) OR
  rockfon OR 
  (usg NEAR ceiling*) OR 
  (usg NEAR "acoustical wall panel*") OR 
  (certainteed NEAR ceiling*) OR 
  "Ceiling Suspension System*"''']})

filters_dict['Cultured Stone'] = pd.DataFrame({"SEARCH": ["Cultured Stone"], "Query": ["""
   (cultured NEAR stone) OR 
   (country NEAR rubble) OR 
   ('old world' NEAR ledge) OR 
   (Stone NEAR Veneer) OR 
   ("Cultured Stone") OR 
   ("Veneer Stone") OR 
   ("Manufactured Stone") OR 
   ("Adhered Stone") OR 
   ("Stone Facing") OR 
   ("Stone Cladding") OR 
   ("Masonry Veneer") OR 
   ("Thin Stone") OR 
   ("Faux Stone") OR 
   ("Artificial Stone") OR 
   ("Simulated Stone") OR 
   ("Architectural Stone") OR 
   ("Cast Stone") OR 
   ("Ledgestone") OR 
   ("Stackstone") OR 
   ("Stacked Stone") OR 
   ("Rubble Stone") OR 
   ("River Rock") OR 
   ("Country Rubble") OR 
   ("Old World Ledge") OR 
   ("Honey Ledge") OR 
   ("Chiseled Stone") OR 
   ("Lompoc") OR 
   ("Palomino") OR 
   ("Tuscan Villa") OR 
   ("Coastal Brown") OR 
   ("Romano") OR 
   ("Woodstone") OR 
   ("Old Country Ledge") OR 
   ("Minnesota Fieldstone") OR 
   ("Urbana Smooth") OR 
   ("Quartzite") OR 
   ("Limestone") OR 
   ("Fieldstone") OR 
   ("Slate") OR 
   ("Sandstone") OR 
   ("Marble") OR 
   ("Clay Unit Masonry") OR 
   ("Concrete Unit Masonry") OR 
   ("CMU") OR 
   ("Spec-Mix Veneer Stone") OR 
   ("EZ Casing") OR 
   (Sill NEAR (Stone OR Masonry OR Lompoc OR Limestone OR Chiseled)) OR 
   ("Light Box" NEAR (Stone OR Masonry OR Lompoc OR Limestone)) OR 
   ("Electrical Box" NEAR (Stone OR Masonry OR Lompoc OR Limestone)) OR
   ('GREY QUARTZIT') OR 
   ('SILL CHISELED') OR 
   ('ELEC BOX LOMPOC SMTH LIM') OR 
   ('LIGHT BOX SMALL LOMPOC') OR 
   ('LIGHT BOX LARGE LOMPOC') OR 
   ('EZ CASING BD') OR 
   ('DOORBELL / HOSE BIB') OR
   ('WOODSTONE / BARN') 
   """
]})


# filters_dict['Drywall'] = pd.DataFrame({"SEARCH": ["Drywall"], "Query": ["""
#    (usg NEAR drywall) OR (usg NEAR gypsum) OR ('united states gypsum' NEAR drywall) OR 
#    (m2tech NEAR drywall) OR (densglass NEAR drywall) OR (denshield NEAR drywall) OR (quietrock NEAR drywall) 
#    OR ('lite type x' NEAR drywall) OR (mold guard NEAR drywall) OR (shaftliner NEAR drywall) OR 
#    ('ultra lightweight' NEAR drywall) OR (SHEETROCK) OR ('MOLD & MOISTURE') OR 
#    (HI ABUSE) OR (DENSGLASS) OR (DENSHIELD) OR ('EASY SAND') OR ('CEMENT BOARD') OR 
#    ('LITE TYPE X') OR ('XP ULTRASHIELD') OR ('FC TYPE X') OR ('FC MOLD GUARD') OR ('LIGHT WEIGHT') 
#    OR ('HI FLEX') OR ('BLUE RIDGE') OR (Durock glass) OR (DUROCK GLASS) OR (DENS SHAFTLINER) OR ('MOLD TOUGH') 
#    OR ('EASY SAND 20MIN') OR ('S/R DW JOINT TAPE') OR ('DURABOND 45MIN') OR ('DURABOND 90MIN') OR 
#    ('QUIETROCK') OR (LITEWEIGHT) OR (Durock) OR (Durock glass)"""]})

filters_dict["Drywall"] = pd.DataFrame({"SEARCH": ["Drywall"], "Query": ["""
                                                                         (
    drywall OR
    gypsum OR "gypsum board" OR "gypsum panel" OR "gypsum wallboard" OR (gypsum W/3 (assembly OR system OR framing OR partition OR ceiling OR sheathing)) OR
    wallboard OR "wall board" OR (wall W/2 board) OR
    sheetrock OR
    plasterboard OR
    "Division 9 Drywall" OR "Division 9 Gypsum" OR "Drywall/Gypsum" OR
    ("backer board" AND (tile OR cement OR gypsum)) OR
    "cement board" OR
    ("RSMeans" W/5 (929 OR 922 OR "Gypsum Board" OR "Supports For Plaster And Gypsum Board")) OR
    ("united states gypsum") OR (USG NEAR/5 (drywall OR gypsum OR board OR panel OR sheetrock)) OR
    "National Gypsum" OR "Georgia-Pacific Gypsum" OR "CertainTeed Gypsum" OR "American Gypsum" OR "Gold Bond" OR
    DensGlass OR DensShield OR DensArmor OR DensDeck OR QuietRock OR Durock OR
    ("Type X" OR "Type C" OR Shaftliner OR "Mold Guard" OR "Mold Tough" OR "Abuse Resistant" OR "Impact Resistant" OR "Fire Rated Gypsum" OR "Moisture Resistant Gypsum")
)"""]})

# filters_dict['Drywall'] = pd.DataFrame({"SEARCH": ["Drywall"], "Query": ["""(
#     drywall OR
#     "gypsum board" OR "gypsum panel" OR "gypsum wallboard" OR (gypsum NEAR/2 board) OR
#     sheetrock OR
#     "Division 9 Drywall" OR "Division 9 Gypsum" OR "Drywall/Gypsum" OR
#     ("RSMeans" W/5 (929 OR 922 OR "Gypsum Board" OR "Supports For Plaster And Gypsum Board")) OR
#     ("united states gypsum") OR (USG NEAR/3 (drywall OR gypsum OR board OR panel OR sheetrock)) OR
#     "National Gypsum" OR "Georgia-Pacific Gypsum" OR "CertainTeed Gypsum" OR "American Gypsum" OR "Gold Bond" OR
#     DensGlass OR DensShield OR DensArmor OR DensDeck OR QuietRock OR Durock OR
#     ("Type X" OR "Type C" OR Shaftliner OR "Mold Guard" OR "Mold Tough" OR "Abuse Resistant" OR "Impact Resistant" OR "Fire Rated Gypsum")
# )"""]})



filters_dict['EIFS'] = pd.DataFrame({"SEARCH": ["EIFS"], "Query": ["""
  (eifs NEAR adex) OR (eifs NEAR dryvit) OR (eifs NEAR sto) OR (eifs NEAR parex) OR
  (stucco NEAR adex) OR (stucco NEAR dryvit) OR (stucco NEAR sto) OR (stucco NEAR parex) OR
  eifs OR 
  "Exterior Insulation and Finish System" OR 
  "Exterior Insulation & Finish System" OR 
  "Exterior Insulation Finish System" OR 
  "Exterior Insulation & Finish S" OR 
  stucco
)                                
"""]})
 
# filters_dict["FRP"] = pd.DataFrame({"SEARCH": ["FRP"], "Query": ["""( acrovyn ) OR
#    ( (frp OR frl) NEAR (wall OR walls OR panel OR panels) ) OR
#    ( "frp panel" OR "frp panels" ) OR
#    ( "fiberglass reinforced panel*" OR "fiberglass reinforced plastic*" OR "fiberglass reinforced laminate*" ) OR
#    ( "fiberglass panel*" NEAR (wall OR walls OR kitchen OR restroom OR sanitary OR corridor) ) OR
#    ( ("construction specialties" OR "cs group") NEAR "wall protection" )"""]})

filters_dict["FRP"] = pd.DataFrame({"SEARCH": ["FRP"], "Query": ["""
( acrovyn ) OR
( (frp OR frl) NEAR (wall OR walls OR panel OR panels) ) OR
( "frp panel" OR "frp panels" ) OR
( "fiberglass reinforced panel*" OR "fiberglass reinforced plastic*" OR "fiberglass reinforced laminate*" ) OR
( "fiberglass panel*" NEAR (wall OR walls OR kitchen OR restroom OR sanitary OR corridor) )"""]})

# filters_dict['Fry Reglet'] = pd.DataFrame({"SEARCH": ["Fry Reglet"], "Query": ["""( FRY NEAR REGLET ) OR
#    ( FRYDRM OR MWRL75C OR MWCL75 OR MWCB75400 OR MWCOSC50 OR DRMF50-50 ) OR
#    ( (PLASTER OR DRYWALL OR GYPSUM OR METAL OR ALUMINUM OR EIFS OR STUCCO OR ACOUSTIC) NEAR (REVEAL OR SCREED OR MOLDING OR REGLET OR TRIM) ) OR
#    ( "CHANNEL SCREED" ) OR
#    ( "DRIP SCREED" ) OR
#    ( 'EIFS DRIP SCRD' ) OR
#    ( 'REG PLASTER CHANNEL' )"""]}) 

filters_dict['Fry Reglet'] = pd.DataFrame({"SEARCH": ["Fry Reglet"], "Query": ["""
                                                                               (
    "FRY REGLET" OR 
    FRYDRM OR MWRL75C OR MWCL75 OR MWCB75400 OR MWCOSC50 OR DRMF50-50 
) OR (
    "ALUMINUM REVEAL" OR "METAL REVEAL" OR 
    "PLASTER REVEAL" OR "DRYWALL REVEAL" OR "GYPSUM REVEAL" OR 
    "WALL REVEAL" OR "CEILING REVEAL" OR
    "EXTRUDED ALUMINUM REVEAL" OR "EXTRUDED ALUMINUM TRIM" OR 
    "CHANNEL SCREED" OR "DRIP SCREED" OR "EIFS DRIP SCRD" OR 
    "ALUMINUM SCREED" OR "METAL SCREED" OR "PLASTER SCREED" OR "STUCCO SCREED" OR "EIFS SCREED" OR
    "ALUMINUM REGLET" OR "METAL REGLET"
) OR (
    (ALUMINUM NEAR (REVEAL OR REGLET OR SCREED)) OR 
    ((DRYWALL OR GYPSUM OR "GYPSUM BOARD" OR GWB OR PLASTER) NEAR (REVEAL OR REGLET)) OR 
    ((PLASTER OR STUCCO OR EIFS) NEAR SCREED) OR 
    ((EIFS OR "EXTERIOR INSULATION") NEAR (REVEAL OR REGLET))
)"""]})

filters_dict['Insulation'] = pd.DataFrame({"SEARCH": ["Insulation"], "Query": ["""
   ('mineral wool') OR (fiberglass) OR ('loose fill') OR (cellulose) OR 
   ('spray foam') OR (rockwool) OR (comforttherm) OR ('kraft faced') OR (unfaced) OR 
   (RAFTRMATE) OR (STYROFOAM) OR (PROPINK) OR (NEXSEAL) OR (APPLEGATE) OR (MetaCaulk) OR 
   (EasySeal) OR (Thermafiber) OR (Barricade) OR (FSK) OR (LAMTEC) OR (Durovent) OR 
   ('FIRE BLOCK') OR (AFB) OR (GREENGRD) OR ('DUST CONTROL') OR ('THERMAX SHEATHING') OR 
   ('SPRAY FOAM') OR ('FIBERGLAS JOINT TAPE') OR ('FIBERGLAS JOINT') OR (WINDLOCK) OR (TYTAN) OR 
   (CERTIFOAM) OR ('BUILDING WRAP') OR (FOAMULAR) OR ('SELECTSOUND BLACK') OR ('NEOPRENE GASKET') 
   OR ('ENERGY SHIELD') OR ('FIRESTRIP') OR ('ABSORPTION PLUS') OR (SAFB) OR (JM) OR 
   (OC) OR (RW) OR (RVLF) OR (ELA) OR (SAU) OR (GSCSF) OR (R) OR (K) OR (U) OR 
   ('FIBEROCK UNDER') OR ('R8U') OR ('R11U') OR ('R19U') OR (R19U)"""]})
                       
# filters_dict['Steel'] = pd.DataFrame({"SEARCH": ["Steel"], "Query": ["""
#    (("STEEL" OR "METAL") AND 
#    (JOIST OR DECKING OR STUD OR TRACK OR CHANNEL OR FURRING OR RUNNER OR ANGLE OR BEAM OR COLUMN OR PLATE OR BAR OR STAIRS OR RAILING OR FABRICATION OR GIRT OR MESH OR LATH OR CLIP OR BRACING OR DOOR OR FRAME OR WINDOW)) 
#    OR ("COLD FORMED") OR ("LIGHT GAUGE") OR ("FLAT STOCK") OR ("PONY WALL") OR (RADIUS) OR 
#    (CURVED) OR (SLOTTED) OR (RESILIENT)"""]})

filters_dict['Steel'] = pd.DataFrame({"SEARCH": ["Steel"], "Query": ["""
(
    ("STEEL" OR "METAL") AND 
    (FRAMING OR JOIST OR DECKING OR STUD OR TRACK OR CHANNEL OR FURRING OR RUNNER OR ANGLE OR BEAM OR COLUMN OR PLATE OR BAR OR STAIRS OR RAILING OR FABRICATION OR GIRT OR BRACING)
) 
OR ("STRUCTURAL STEEL") 
OR ("COLD FORMED") 
OR ("LIGHT GAUGE") 
OR ("METAL STUD") 
OR ("STEEL STUD")                                             
"""]})

# Core search terms comprised of drywall, ceiling, and steel
drywall_query = filters_dict['Drywall']['Query'].values[0]
ceiling_query = filters_dict['Ceilings']['Query'].values[0]
steel_query = filters_dict['Steel']['Query'].values[0]
filters_dict['Core'] = pd.DataFrame({"SEARCH": ['Core'], "Query": [f"""{drywall_query} OR {ceiling_query} OR {steel_query}"""]})
                       
# Create a dictionary to hold the DataFrames filtered by search name
filters_df_dict = {} 
for search_name in filters_dict.keys(): 
    filters_df_dict[search_name] = sampled_df[sampled_df['SEARCH'] == search_name]

In [81]:
from search_eval_utils import prompt_v1

os.makedirs('predictions/filters', exist_ok=True)
searches = ["Ceilings", "Core", "Cultured Stone", "Drywall", "EIFS", "FRP", "Fry Reglet", "Insulation", "Steel"]
# searches=["Fry Reglet"]

for search_name in searches: 
    print(f"Processing search: {search_name}")
    process_and_save_prompt_output(gemini_client=gemini_client, 
                                   model_id=MODEL_ID, 
                                   prompt_function=prompt_v1,
                                   df=filters_df_dict[search_name].drop(columns=['True_Search_Label']), # Fetch the DataFrame for the current search
                                   searches_df=filters_dict[search_name],   # Fetch the corresponding search term and query
                                   query_column='Query', 
                                   output_filename=f'predictions/filters/{search_name}.jsonl')

Processing search: Ceilings


100%|██████████| 20/20 [00:23<00:00,  1.15s/it]


Processing search: Core


100%|██████████| 20/20 [00:24<00:00,  1.21s/it]


Processing search: Cultured Stone


...

100%|██████████| 20/20 [00:25<00:00,  1.27s/it]


Processing search: Insulation


100%|██████████| 20/20 [00:20<00:00,  1.03s/it]


Processing search: Steel


100%|██████████| 20/20 [00:22<00:00,  1.13s/it]


In [82]:
from sklearn.preprocessing import LabelBinarizer 
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn import metrics
import numpy as np

searches = ["Ceilings", "Core", "Cultured Stone", "Drywall", "EIFS", "FRP", "Fry Reglet", "Insulation", "Steel"]

# Dictionary for storing all responses for each search
all_responses_df = {} 

# Iterate through each search name and load the responses
for search_name in searches: 
    print("-----------------------\nSEARCH", search_name)

    # Read results
    responses_df = filters_df_dict[search_name].copy()
    version_responses = ['v1_stratified', f'filters/{search_name}']

    # Load responses for original filter and new filter
    for version in version_responses:
        version_df = jsonl_to_df(f'predictions/{version}.jsonl')
        version_df.rename({'Project_related_to_Search': f'Project_related_to_Search_{version}', 'Reasoning': f'Reasoning_{version}'}, axis=1, inplace=True)
        responses_df = responses_df.merge(version_df, on=['ProjectID','SEARCH'], how='left')

    # Keep only the relevant columns
    related_cols = responses_df.filter(regex='related|True_Search_Label').columns
    responses_df.dropna(subset=related_cols, inplace=True)

    # Add the dataframe results to dictionary 
    all_responses_df[search_name] = responses_df

    # Encode YES/NO labels
    lb = LabelBinarizer()
    lb.fit(["NO", "YES"])
    for col in related_cols: 
        responses_df[f"{col}_label"] = lb.transform(responses_df[col])

    related_cols_labels = [f"{col}_label" for col in related_cols]

    # Print metrics for original and new filters
    for col in related_cols_labels:
        if col != 'True_Search_Label_label': 
            
            if 'filters' in col: 
                print("Updated filter")
            else: 
                print("Original filter ")

            # Calculate metrics
            f1_overall = f1_score(responses_df['True_Search_Label_label'], responses_df[col], zero_division=np.nan)
            precision_overall = precision_score(responses_df['True_Search_Label_label'], responses_df[col], zero_division=np.nan)
            recall_overall = recall_score(responses_df['True_Search_Label_label'], responses_df[col], zero_division=np.nan) 

            # print metrics
            print(f"f1: {f1_overall}, precision: {precision_overall}, recall:{recall_overall} ")


-----------------------
SEARCH Ceilings
Original filter 
f1: 0.5833333333333334, precision: 0.5, recall:0.7 
Updated filter
f1: 0.56, precision: 0.4666666666666667, recall:0.7 
-----------------------
SEARCH Core
Original filter 
f1: 0.6666666666666666, precision: 0.5, recall:1.0 
Updated filter
f1: 0.6666666666666666, precision: 0.5, recall:1.0 
-----------------------
SEARCH Cultured Stone
Original filter 
f1: 0.26666666666666666, precision: 0.4, recall:0.2 
Updated filter
f1: 0.35294117647058826, precision: 0.42857142857142855, recall:0.3 
-----------------------
SEARCH Drywall
Original filter 
f1: 0.6428571428571429, precision: 0.5, recall:0.9 
Updated filter
f1: 0.6206896551724138, precision: 0.47368421052631576, recall:0.9 
-----------------------
SEARCH EIFS
Original filter 
f1: 0.0, precision: 0.0, recall:0.0 
Updated filter
f1: 0.5217391304347826, precision: 0.46153846153846156, recall:0.6 
-----------------------
SEARCH FRP
Original filter 
f1: 0.0, precision: 0.0, recall:0.0

In [83]:
name = "Fry Reglet"
print(all_responses_df[name][['ProjectID', 'True_Search_Label_label', 
                              f'Project_related_to_Search_filters/{name}_label', 
                              f'Reasoning_filters/{name}']].to_markdown())

|    |   ProjectID |   True_Search_Label_label |   Project_related_to_Search_filters/Fry Reglet_label | Reasoning_filters/Fry Reglet                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |
|---:|------------:|--------------------------:|-----------------------------------------------------:|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [84]:
# Create DataFrame for new filteres
new_filters = pd.DataFrame()

for key, df in filters_dict.items(): 
    # Get the search term and query
    search_term = df['SEARCH'].values[0]
    query = df['Query'].values[0]
    # Create a new DataFrame with the search term and new query
    new_filters = pd.concat([new_filters, pd.DataFrame({"SEARCH": [search_term], "NEW_FILTER": [query]})], ignore_index=True)
    
new_filters

,SEARCH,NEW_FILTER
0,Ceilings,\n( Title:(ceiling* AND (replacement OR repair...
1,Cultured Stone,\n (cultured NEAR stone) OR \n (country NE...
2,Drywall,\n ...
3,EIFS,\n (eifs NEAR adex) OR (eifs NEAR dryvit) OR ...
4,FRP,\n( acrovyn ) OR\n( (frp OR frl) NEAR (wall OR...
5,Fry Reglet,\n ...
6,Insulation,\n ('mineral wool') OR (fiberglass) OR ('loo...
7,Steel,"\n(\n (""STEEL"" OR ""METAL"") AND \n (FRAMI..."
8,Core,\n ...


In [85]:
searches_df = pd.read_csv(SEARCHES_DATASET_NAME)
searches_df = searches_df[['SEARCH', 'FILTER']]

# Add new filters to searches_df
searches_df = searches_df.merge(new_filters, how = 'left', on = 'SEARCH')

keep_original_filters = []
for query in keep_original_filters: 
    searches_df.loc[searches_df['SEARCH'] == query, 'NEW_FILTER'] = searches_df.loc[searches_df['SEARCH'] == query, 'FILTER']

# Save the updated searches_df
searches_df.to_csv(SEARCHES_DATASET_NAME, index=False)

## Run Total Evaluation on new filters and compare with old filters

In [87]:
# New filters
from search_eval_utils import prompt_v1
process_and_save_prompt_output(gemini_client=gemini_client, 
                               model_id=MODEL_ID, 
                               prompt_function=prompt_v1, 
                               df=sampled_df_no_label, 
                               searches_df=pd.read_csv(SEARCHES_DATASET_NAME),
                               query_column='NEW_FILTER', 
                               output_filename='predictions/v1_stratified_new_filter.jsonl')

  0%|          | 0/172 [00:00<?, ?it/s]

100%|██████████| 172/172 [03:21<00:00,  1.17s/it]


In [88]:
from sklearn.preprocessing import LabelBinarizer 

# Read results
responses_df = sampled_df.copy()  
version_responses = ['v1_stratified', 'v1_stratified_new_filter']

# Load df versions 
for version in version_responses:
    version_df = jsonl_to_df(f'predictions/{version}.jsonl')
    version_df.rename({'Project_related_to_Search': f'Project_related_to_Search_{version}', 
                       'Reasoning': f'Reasoning_{version}'}, axis=1, inplace=True)
    responses_df = responses_df.merge(version_df, on=['ProjectID','SEARCH'], how='left')

related_cols = responses_df.filter(regex='related|True_Search_Label').columns
responses_df.dropna(subset=related_cols, inplace=True)

# Encode the relevance columns 
lb = LabelBinarizer()
lb.fit(["NO", "YES"])
for col in related_cols: 
    responses_df[f"{col}_label"] = lb.transform(responses_df[col])

In [89]:
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn import metrics
import numpy as np
from tabulate import tabulate

related_cols_labels = [f"{col}_label" for col in related_cols] # add _label suffix
search_cols = responses_df['SEARCH'].unique()

# Print metrics for each version
for col in related_cols_labels: 
    if col != 'True_Search_Label_label': 

        print(col) 
        # Init metrics table
        metrics_data = [] 
        headers = ["Search", "F1", "Precision", "Recall"]

        # Calculate and add overall metrics
        f1_overall = f1_score(responses_df['True_Search_Label_label'], responses_df[col], zero_division=np.nan)
        precision_overall = precision_score(responses_df['True_Search_Label_label'], responses_df[col], zero_division=np.nan)
        recall_overall = recall_score(responses_df['True_Search_Label_label'], responses_df[col], zero_division=np.nan) 
        metrics_data.append(["Overall", f1_overall, precision_overall, recall_overall])

        # Calculate and add metrics for each search
        for search in search_cols:

            # Filter the DataFrame for the current search
            search_df = responses_df[responses_df['SEARCH'] == search]
            
            # Calculate metrics for the current search
            f1 = f1_score(search_df['True_Search_Label_label'], search_df[col], zero_division=np.nan)
            precision = precision_score(search_df['True_Search_Label_label'], search_df[col], zero_division=np.nan)
            recall = recall_score(search_df['True_Search_Label_label'], search_df[col], zero_division=np.nan)
            metrics_data.append([search, f1, precision, recall])
        
        # Print the metrics table
        table = tabulate(metrics_data, headers=headers, tablefmt="grid")
        print(table, "\n")

Project_related_to_Search_v1_stratified_label
+----------------+----------+-------------+----------+
| Search         |       F1 |   Precision |   Recall |
+================+==========+=============+==========+
| Overall        | 0.535519 |    0.485149 | 0.597561 |
+----------------+----------+-------------+----------+
| Ceilings       | 0.583333 |    0.5      | 0.7      |
+----------------+----------+-------------+----------+
| Core           | 0.666667 |    0.5      | 1        |
+----------------+----------+-------------+----------+
| Cultured Stone | 0.266667 |    0.4      | 0.2      |
+----------------+----------+-------------+----------+
| Drywall        | 0.642857 |    0.5      | 0.9      |
+----------------+----------+-------------+----------+
| EIFS           | 0        |    0        | 0        |
+----------------+----------+-------------+----------+
| FRP            | 0        |    0        | 0        |
+----------------+----------+-------------+----------+
| Fry Reglet     | 